![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 2 — Ingeniería de Datos
**Rol:** CRB_DATA_ANALYTICS | **Tiempo:** 15 min | **Criterio:** Pipeline declarativo, calidad, observabilidad, CI/CD, linaje

In [ ]:
USE ROLE CRB_DATA_ANALYTICS;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 0 — Lanza estos prompts en CoCo AHORA (mientras ejecutas el resto)
Copia **ambos prompts** en Cortex Code al inicio del track. CoCo los ejecuta en paralelo mientras tú avanzas con los bloques 1 y 2. Cuando llegues al bloque 3, los resultados ya estarán listos.

---
### Prompt 1: API TRM Colombia (Superfinanciera)
Copia y pega en CoCo:

In [ ]:
-- PROMPT PARA COCO (copiar todo este bloque y pegarlo en Cortex Code):
--
-- Crea un pipeline completo en la cuenta credibanco-hol que consulte la API
-- publica de la Superfinanciera de Colombia para traer la TRM del ultimo ano.
-- API: https://www.datos.gov.co/resource/mcec-87by.json
-- Parametros: $limit=365, $order=vigenciadesde DESC
-- No requiere autenticacion.
-- Necesito:
-- 1. Network Rule + External Access Integration para datos.gov.co
-- 2. UDF en Python que llame la API y retorne los registros como tabla
-- 3. Tabla CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA (fecha DATE, trm NUMBER(10,2))
-- 4. Cargar el historico del ultimo ano
-- 5. Un Task programado diario a las 8AM Colombia (CRON '0 13 * * *' UTC)
-- 6. Verificar con: SELECT * FROM TRM_HISTORICA ORDER BY fecha DESC LIMIT 5
-- Ejecuta todo en paralelo, no esperes confirmacion entre pasos.

### Prompt 2: Openflow S3 → Snowflake
Copia y pega en CoCo (puedes cambiar el nombre de la tabla destino):

In [ ]:
-- PROMPT PARA COCO (copiar todo este bloque y pegarlo en Cortex Code):
--
-- Configura un pipeline que ingeste datos desde el bucket S3 publico
-- s3://sfquickstarts/summit_dev_day_2026_automated_intelligence_hol/
-- hacia Snowflake. Paraleliza todo lo que puedas.
--
-- Necesito:
-- 1. External stage apuntando al S3 (publico, sin credenciales AWS)
-- 2. Inferir el schema de customers.parquet con INFER_SCHEMA
-- 3. Crear la tabla destino con USING TEMPLATE
-- 4. COPY INTO para cargar los datos (2M filas de customers)
-- 5. Verificar conteo y mostrar 5 filas de ejemplo
--
-- >>> PERSONALIZA EL NOMBRE DE LA TABLA DESTINO AQUI <<<
-- Tabla destino: CREDIBANCO_HOL.PLATAFORMA.CUSTOMERS_OPENFLOW
-- (cambia el nombre si quieres, por ejemplo: MI_TABLA_S3, DATOS_EXTERNOS, etc.)
--
-- Usa la conexion credibanco-hol. Ejecuta todo sin esperar confirmacion.

---
**Mientras CoCo trabaja en los prompts anteriores, continúa ejecutando los bloques 1 y 2 abajo.**
Cuando llegues al bloque 3, los datos de la TRM y del S3 ya estarán en tus tablas.

---

## Bloque 1 — Evidencia: Pipeline declarativo sin Spark

In [ ]:
-- Dynamic Tables: pipeline bronce→plata→oro SIN orquestador
SHOW DYNAMIC TABLES IN DATABASE CREDIBANCO_HOL;

In [ ]:
-- Ver datos transformados en la capa Silver
SELECT COUNT(*) AS filas_silver FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER;
SELECT * FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER LIMIT 5;

In [ ]:
-- Linaje nativo: trazar la cadena end-to-end
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER', 'table', 'upstream', 3
));

## Bloque 2 — Ejecutar: Crear nueva DT + ver propagación

In [ ]:
-- Crear una Dynamic Table nueva: agregados por hora
CREATE OR REPLACE DYNAMIC TABLE CREDIBANCO_HOL.PAGOS.DT_HOURLY_<TU_USUARIO>
  TARGET_LAG = '5 minutes'
  WAREHOUSE = CREDIBANCO_HOL_WH
AS
SELECT DATE_TRUNC('hour', FECHA_HORA) AS hora,
       CIUDAD, COUNT(*) AS num_tx, SUM(MONTO) AS monto_total
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
GROUP BY 1, 2;

In [ ]:
-- Verificar refresh automático
SHOW DYNAMIC TABLES LIKE 'DT_HOURLY%' IN SCHEMA CREDIBANCO_HOL.PAGOS;

In [ ]:
-- Insertar dato nuevo y ver propagación
INSERT INTO CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
SELECT 999999, 1, 1, '4111111111111111', '5411', 'Bogota',
       CURRENT_TIMESTAMP(), 9999999, '00', 'ECOMMERCE';

## Bloque 3 — Ingesta desde API externa (TRM Colombia)
Conectamos Snowflake a la API pública de la Superfinanciera para traer la TRM del último año. Todo se ejecuta dentro de Snowflake — sin herramientas externas.

In [ ]:
-- Paso 1: Crear Network Rule para permitir acceso a datos.gov.co
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE CREDIBANCO_HOL.PLATAFORMA.NR_DATOS_GOV
  MODE = EGRESS TYPE = HOST_PORT
  VALUE_LIST = ('www.datos.gov.co:443');

In [ ]:
-- Paso 2: External Access Integration
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION CREDIBANCO_HOL.PLATAFORMA.EAI_DATOS_GOV
  ALLOWED_NETWORK_RULES = (CREDIBANCO_HOL.PLATAFORMA.NR_DATOS_GOV)
  ENABLED = TRUE;

In [ ]:
-- Paso 3: UDF Python que consulta la API y retorna la TRM
CREATE OR REPLACE FUNCTION CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(dias INT)
  RETURNS TABLE (fecha DATE, trm FLOAT)
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.9'
  PACKAGES = ('requests')
  EXTERNAL_ACCESS_INTEGRATIONS = (EAI_DATOS_GOV)
  HANDLER = 'run'
AS $$
import requests
def run(dias):
    url = f'https://www.datos.gov.co/resource/mcec-87by.json?$limit={dias}&$order=vigenciadesde DESC'
    r = requests.get(url)
    data = r.json()
    for row in data:
        yield (row['vigenciadesde'][:10], float(row['valor']))
$$;

In [ ]:
-- Paso 4: Consultar la TRM del último año (datos REALES de la Superfinanciera)
SELECT * FROM TABLE(CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(30))
ORDER BY fecha DESC;

In [ ]:
-- Paso 5: Almacenar en tabla persistente
CREATE OR REPLACE TABLE CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA AS
SELECT * FROM TABLE(CREDIBANCO_HOL.PLATAFORMA.GET_TRM_HISTORICA(365));

In [ ]:
-- Paso 6: Verificar datos cargados
SELECT COUNT(*) AS dias, MIN(fecha) AS desde, MAX(fecha) AS hasta,
       ROUND(AVG(trm), 2) AS trm_promedio
FROM CREDIBANCO_HOL.PLATAFORMA.TRM_HISTORICA;

### Prompt CoCo (opcional)
Si quieres que CoCo además programe la actualización diaria, copia este prompt:

> **Crea un Task llamado TASK_TRM_DIARIA que ejecute la función GET_TRM_HISTORICA todos los días a las 8AM Colombia (UTC-5) y haga MERGE INTO TRM_HISTORICA para agregar solo los días nuevos. Usa CRON '0 13 * * *' (8AM COT = 1PM UTC).**

In [ ]:
-- Verificación final
SELECT 'T2_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_<TU_USUARIO>) AS filas_nueva_dt;